# Module 7.1 — Deep Q-Networks (DQN)

Every module so far has been **supervised learning**: given labeled examples, learn to predict. Reinforcement learning is different. There are no labels — only an agent, an environment, and a reward signal. The agent has to figure out what to do by trying things and seeing what happens.

This is how DeepMind's DQN learned to play Atari games from raw pixels in 2013, and the same architecture underpins more recent systems.

**What you'll learn:**
- The RL framework: agent, environment, state, action, reward, episode
- Q-learning: assigning a value to (state, action) pairs
- The Bellman equation: the recursive relationship that makes Q-learning work
- DQN's two key tricks: experience replay and the target network
- Train a DQN to balance a pole on CartPole — same algorithm that plays Atari, just with a simpler state

**Requires:** `pip install gymnasium`

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import matplotlib.pyplot as plt
from collections import deque
import gymnasium as gym

print(f'PyTorch version: {torch.__version__}')
print(f'Gymnasium version: {gym.__version__}')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. The Reinforcement Learning Framework

RL is structured as a loop between an **agent** and an **environment**:

```
         action a_t
Agent  ─────────────►  Environment
       ◄─────────────
       state s_{t+1}
       reward r_t
```

At each timestep t:
1. The agent observes the current **state** s_t
2. It picks an **action** a_t according to its **policy** π(s)
3. The environment returns a **reward** r_t and the next state s_{t+1}
4. Repeat until the episode ends (pole falls, game over, etc.)

**The goal:** learn a policy that maximises total discounted reward over an episode:

```
G_t = r_t + γ·r_{t+1} + γ²·r_{t+2} + ...
```

γ (gamma) is the **discount factor** (0 < γ < 1). A reward now is worth more than a reward later — both because the future is uncertain and because we want the agent to be efficient.

### Key terms

| Term | Meaning |
|---|---|
| **State** s | Everything the agent can observe about the world |
| **Action** a | A choice the agent makes |
| **Reward** r | A scalar signal: positive = good, negative = bad |
| **Policy** π | The agent's strategy: given state s, pick action a |
| **Episode** | One complete run from start to terminal state |
| **Return** G | Total discounted reward from a given timestep |

### Supervised vs reinforcement learning

In supervised learning you're told the correct answer. In RL you only get a reward — and often delayed: a chess engine doesn't know if a move was good until many moves later. This **credit assignment problem** is one of the core challenges in RL.

## 3. The CartPole Environment

CartPole-v1 is the "hello world" of RL. A pole is balanced on a cart that can move left or right. The goal is to keep the pole from falling.

- **State:** 4 numbers — cart position, cart velocity, pole angle, pole angular velocity
- **Actions:** 2 — push left (0) or push right (1)
- **Reward:** +1 for every timestep the pole stays upright
- **Episode ends:** pole tips >15°, cart moves >2.4 units, or 500 steps reached
- **Solved:** average reward ≥ 475 over 100 consecutive episodes

In [ ]:
env = gym.make('CartPole-v1')

n_obs     = env.observation_space.shape[0]
n_actions = env.action_space.n
print(f'Observation space: {env.observation_space}  ({n_obs} floats)')
print(f'Action space:      {env.action_space}  ({n_actions} discrete actions)')

# See what one step looks like
state, _ = env.reset(seed=SEED)
print(f'\nInitial state: {state}')
print('  [cart_pos, cart_vel, pole_angle, pole_ang_vel]')

next_state, reward, terminated, truncated, info = env.step(1)  # push right
print(f'After pushing right: {next_state}')
print(f'Reward: {reward},  terminated: {terminated}')

In [ ]:
# Baseline: random policy
def run_episode(env, policy_fn, seed=None):
    state, _ = env.reset(seed=seed)
    total_reward = 0
    for _ in range(500):
        action = policy_fn(state)
        state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break
    return total_reward


random_policy = lambda s: env.action_space.sample()
random_rewards = [run_episode(env, random_policy) for _ in range(200)]

print(f'Random policy — mean reward: {np.mean(random_rewards):.1f}, '
      f'max: {np.max(random_rewards):.0f}  (solved = 475)')

## 4. Q-Learning and the Bellman Equation

### The Q-function

Instead of learning a policy directly, Q-learning learns a **value function**:

```
Q(s, a) = expected total discounted reward if we take action a in state s
          and then act optimally afterwards
```

Once we have Q, the optimal policy is trivial: always take the action with the highest Q-value.

### The Bellman equation

How do we learn Q? It satisfies a recursive consistency condition:

```
Q(s, a) = r  +  γ · max_{a'} Q(s', a')
           ↑         ↑
     immediate    best possible future value
     reward       from the next state
```

This is the **Bellman equation**. It says: the value of (s, a) equals the immediate reward plus the discounted value of the best next action. If our Q estimates are self-consistent under this equation, they're correct.

### Tabular Q-learning (and why it breaks)

Classic Q-learning stores Q in a table and updates it iteratively:
```
Q(s, a) ← Q(s, a) + α · [r + γ · max_{a'} Q(s', a')  −  Q(s, a)]
```

This works for small discrete state spaces (like a grid world), but CartPole has **continuous** state (4 floats) and Atari has 210×160×3 pixel observations — far too many states to enumerate. We need a function approximator. Enter the neural network.

## 5. DQN: Two Key Tricks

DQN (Mnih et al., 2013/2015) replaces the Q-table with a neural network `Q(s, a; θ)`. The output is a Q-value for every action simultaneously — one forward pass covers all actions for a given state.

Naive "just train a network on the Bellman equation" doesn't work well. DQN adds two fixes:

### Trick 1: Experience Replay

**Problem:** consecutive (s, a, r, s') tuples are highly correlated — the agent is in one region of the state space, then nearby regions. SGD assumes i.i.d. mini-batches; correlated data causes the network to forget what it learned about other states.

**Fix:** store transitions in a **replay buffer** (a large circular queue). At each training step, sample a random mini-batch from the buffer. This breaks the temporal correlations and reuses each experience multiple times.

### Trick 2: Target Network

**Problem:** the Bellman update target `r + γ · max_{a'} Q(s', a'; θ)` uses the same network θ we're trying to update. Every gradient step shifts the target, creating a moving goalpost — training is unstable.

**Fix:** maintain a second **target network** with weights θ⁻, a periodic copy of θ. Use θ⁻ to compute targets, update only θ. Every N steps, copy θ → θ⁻. The target is now stable for N steps at a time.

## 6. Implementation

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states),      dtype=torch.float32).to(device),
            torch.tensor(actions,               dtype=torch.long).to(device),
            torch.tensor(rewards,               dtype=torch.float32).to(device),
            torch.tensor(np.array(next_states), dtype=torch.float32).to(device),
            torch.tensor(dones,                 dtype=torch.float32).to(device),
        )

    def __len__(self):
        return len(self.buffer)

In [ ]:
class DQN(nn.Module):
    def __init__(self, n_obs, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_obs, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),  # one Q-value per action
        )

    def forward(self, x):
        return self.net(x)  # [B, n_actions]


policy_net = DQN(n_obs, n_actions).to(device)
target_net = DQN(n_obs, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())  # identical weights at start
target_net.eval()  # target net is never trained directly

print(policy_net)
print(f'\nParameters: {sum(p.numel() for p in policy_net.parameters()):,}')
print()
print('Output for a random state (one Q-value per action):')
dummy = torch.randn(1, n_obs).to(device)
with torch.no_grad():
    q_vals = policy_net(dummy)
print(f'  Q(s, push_left)  = {q_vals[0, 0].item():.4f}')
print(f'  Q(s, push_right) = {q_vals[0, 1].item():.4f}')

In [ ]:
# Hyperparameters
BUFFER_SIZE   = 10_000   # replay buffer capacity
BATCH_SIZE    = 64       # mini-batch size for each gradient step
GAMMA         = 0.99     # discount factor
LR            = 1e-4     # Adam learning rate
EPS_START     = 1.0      # epsilon-greedy: start fully random
EPS_END       = 0.05     # floor — always keep some exploration
EPS_DECAY     = 0.995    # multiply epsilon by this each episode
TARGET_UPDATE = 20       # copy policy_net -> target_net every N episodes
N_EPISODES    = 600

optimizer    = optim.Adam(policy_net.parameters(), lr=LR)
replay_buf   = ReplayBuffer(BUFFER_SIZE)
epsilon      = EPS_START

In [ ]:
def select_action(state, epsilon):
    if random.random() < epsilon:
        return env.action_space.sample()          # explore
    with torch.no_grad():
        s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        return policy_net(s).argmax(dim=1).item() # exploit


def optimize():
    if len(replay_buf) < BATCH_SIZE:
        return None

    states, actions, rewards, next_states, dones = replay_buf.sample(BATCH_SIZE)

    # Q(s, a) for the actions actually taken
    q_current = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    # Bellman target: r + γ · max_{a'} Q_target(s', a')   (0 if terminal)
    with torch.no_grad():
        q_next  = target_net(next_states).max(dim=1).values
        q_target = rewards + GAMMA * q_next * (1.0 - dones)

    # Huber loss is more robust to outliers than MSE for RL
    loss = nn.SmoothL1Loss()(q_current, q_target)
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(policy_net.parameters(), max_norm=10)  # gradient clipping
    optimizer.step()
    return loss.item()

## 7. Training

The full loop each episode:
1. Reset the environment
2. For each step: pick action (ε-greedy), step env, store transition in replay buffer, optimize
3. Decay ε — shift from exploration towards exploitation as training progresses
4. Every N episodes, copy policy_net → target_net

Watch the 100-episode moving average — when it crosses ~475 the agent has learned to balance the pole.

In [ ]:
episode_rewards = []

for episode in range(1, N_EPISODES + 1):
    state, _ = env.reset()
    total_reward = 0

    for _ in range(500):
        action = select_action(state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        replay_buf.push(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        optimize()

        if done:
            break

    epsilon = max(EPS_END, epsilon * EPS_DECAY)
    episode_rewards.append(total_reward)

    if episode % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if episode % 50 == 0:
        avg = np.mean(episode_rewards[-100:])
        print(f'Episode {episode:4d} | Reward: {total_reward:5.0f} | '
              f'Avg(100): {avg:6.1f} | epsilon: {epsilon:.3f}')

env.close()

## 8. Results

In [ ]:
window = 100
moving_avg = [np.mean(episode_rewards[max(0, i - window):i + 1])
              for i in range(len(episode_rewards))]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(episode_rewards, alpha=0.3, color='steelblue', label='Episode reward')
ax.plot(moving_avg, color='steelblue', linewidth=2, label=f'{window}-episode average')
ax.axhline(y=475, color='green', linestyle='--', linewidth=1.5, label='Solved threshold (475)')
ax.axhline(y=np.mean(random_rewards), color='red', linestyle=':', linewidth=1.5, label='Random policy baseline')
ax.set_xlabel('Episode')
ax.set_ylabel('Total reward')
ax.set_title('DQN training on CartPole-v1')
ax.legend()
plt.tight_layout()
plt.show()

final_avg = np.mean(episode_rewards[-100:])
print(f'Final 100-episode average: {final_avg:.1f}  (solved = 475)')

In [ ]:
# Evaluate the trained agent (greedy policy, no exploration)
eval_env = gym.make('CartPole-v1')
greedy   = lambda s: policy_net(torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device)).argmax().item()

eval_rewards = []
with torch.no_grad():
    for i in range(100):
        eval_rewards.append(run_episode(eval_env, greedy, seed=i))
eval_env.close()

print(f'Trained DQN — mean: {np.mean(eval_rewards):.1f}, '
      f'median: {np.median(eval_rewards):.1f}, '
      f'min: {np.min(eval_rewards):.0f}')
print(f'Random policy — mean: {np.mean(random_rewards):.1f}')
print(f'Improvement: {np.mean(eval_rewards) / np.mean(random_rewards):.1f}x')

## 9. What Changes for Atari and Chess

CartPole gives us 4 clean numbers as the state. Real games are harder:

### Atari (DQN, 2013)

State = raw pixels (210×160×3). The DQN paper used a CNN instead of the MLP we built here:

```
4 stacked grayscale frames (84×84×4)
  -> Conv(32, k=8, s=4) + ReLU
  -> Conv(64, k=4, s=2) + ReLU  
  -> Conv(64, k=3, s=1) + ReLU
  -> Flatten -> Linear(512) -> ReLU
  -> Linear(n_actions)
```

Four frames are stacked so the network can infer velocity (one frame can't tell if a ball is moving left or right). Everything else — replay buffer, target network, ε-greedy, Bellman update — is identical to what we just built.

### Chess / Go (AlphaZero, 2017)

DQN doesn't work well for chess because:
- The action space is huge (~1800 legal moves per position on average)
- Rewards are entirely delayed — you only find out you won or lost at the end of the game
- The optimal move isn't just "highest Q-value now" — you need to think ahead

AlphaZero solves this with a different approach:
1. A neural network outputs *both* a **policy** (probability over moves) and a **value** (who's winning)
2. **Monte Carlo Tree Search** uses the network's outputs to guide lookahead search
3. The network is trained purely from **self-play** — no human game data

Module 7.3 builds this from scratch on Tic-tac-toe.

In [ ]:
# The Atari CNN architecture (reference — not run here, needs ALE)
class AtariDQN(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4), nn.ReLU(),  # 4 stacked frames
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 512), nn.ReLU(),
            nn.Linear(512, n_actions),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


atari_net = AtariDQN(n_actions=18)  # 18 possible actions in most Atari games
atari_params = sum(p.numel() for p in atari_net.parameters())
cartpole_params = sum(p.numel() for p in policy_net.parameters())
print(f'CartPole DQN (MLP):  {cartpole_params:,} parameters')
print(f'Atari DQN (CNN):     {atari_params:,} parameters')
print()
print('Same training loop — just swap the network and the environment.')

## Summary

| Concept | Key takeaway |
|---|---|
| RL framework | Agent takes actions, receives rewards; goal is to maximise discounted return |
| Q-function | Q(s, a) = expected return if we take action a in state s, then act optimally |
| Bellman equation | Q(s,a) = r + γ · max Q(s', a') — recursive self-consistency condition |
| DQN | Neural network approximates Q; same algorithm scales from CartPole to Atari |
| Experience replay | Random mini-batches from a buffer break temporal correlations |
| Target network | Frozen copy of weights stabilises the training target |
| ε-greedy | Balance exploration (random) vs exploitation (greedy) during training |

**What's next:**
- **7.2 — Policy Gradients:** instead of learning Q-values, directly learn a policy π(a|s). Works for continuous action spaces (e.g. controlling a robot joint angle) where DQN breaks down.
- **7.3 — AlphaZero:** combines a policy+value network with lookahead search. The idea that beat the world champion at Go with zero human game data.